# Neanderthal Ancestry and Cancer Risk — Analysis

This notebook contains the code necessary to conduct all data analyses for the
Neanderthal Ancestry and Cancer Risk project. Sections follow the order of
appearance in the manuscript, and their numbers match the companion
`figures.ipynb`. Use the Contents below to navigate.

Most results are produced by the pipeline on Wynton and are read here from
their canonical output files rather than recomputed. The purpose of this
notebook is largely traceability. Section 12 checks every manuscript-quoted
value against the pipeline outputs in one block.

Tables this notebook writes are named by content rather than by table number,
so renumbering the supplement does not break them. Shared helpers live in
`utils.py`, beside this notebook. Run this notebook before `figures.ipynb`,
which reads several of its outputs.

## Contents

- [1. Setup](#1.-Setup)
- [2. Cancer GWAS](#2.-Cancer-GWAS)
- [3. Neanderthal variant maps](#3.-Neanderthal-variant-maps)
- [4. Neanderthal variant MAFs](#4.-Neanderthal-variant-MAFs)
- [5. GWAS loci with Neanderthal variants](#5.-GWAS-loci-with-Neanderthal-variants)
- [6. Fine-mapping](#6.-Fine-mapping)
- [7. Neanderthal-majority signals](#7.-Neanderthal-majority-signals)
- [8. Adaptive introgression](#8.-Adaptive-introgression)
- [9. Locus examples](#9.-Locus-examples)
- [10. Ancient DNA](#10.-Ancient-DNA)
- [11. Heritability](#11.-Heritability)
- [12. Manuscript values](#12.-Manuscript-values)
- [13. Session information](#13.-Session-information)

## 1. Setup

Imports, paths and the canonical pipeline outputs every section below reads
from. The inventory printed at the end of this section records what each file
contained when the notebook was last run, so a stale input is visible here
rather than as an unexplained change in a downstream number.

In [1]:
import platform
import re
import sys
from itertools import combinations
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
import pybedtools as pbt
import yaml

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)

In [2]:
project_directory = Path(
    '/wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk'
)
assert project_directory.exists(), f'{project_directory} is not reachable'

notebooks_directory = project_directory / 'scripts' / 'notebooks'
sys.path.insert(0, str(notebooks_directory))

from utils import (  # noqa: E402
    clean_cancer_names,
    get_credible_set,
    intersect_with_cres,
    prefer_current,
    summarize_phenotypes,
)

In [3]:
data_directory = project_directory / 'data'
adaptive_introgression_directory = data_directory / 'adaptive_introgression'
ancient_genomes_directory = data_directory / 'ancient_genomes'
annotations_directory = data_directory / 'annotations'
fine_mapping_directory = data_directory / 'fine_mapping'
introgressed_variants_directory = data_directory / 'introgressed_variants'
ldsc_directory = (
    data_directory / 'LD_score_regression' / 'results' / 'summaries'
)
metadata_directory = data_directory / 'metadata'
open_targets_directory = data_directory / 'Open_Targets'
significant_gwas_variants_directory = (
    data_directory / 'significant_GWAS_variants'
)

# Every table this notebook writes goes here; figures.ipynb reads from it.
supplementary_tables_directory = data_directory / 'supplementary_tables'
supplementary_tables_directory.mkdir(exist_ok=True)

In [4]:
# The fine-mapping pipeline writes `_current` files beside the stable ones
# after a partial rerun, and the R figure helpers prefer them when present,
# so every fine-mapping input goes through prefer_current().
fine_mapping_files = {
    'Fine-mapping loci': 'fine_mapping_loci.txt',
    'Locus fit status': 'fit_status.txt',
    'Locus summary': 'locus_summary.txt',
    'Credible set variants': 'credible_set_variants.txt',
    'Selected credible sets': 'credible_set_summary_selected.txt',
    'Deduplicated credible sets': 'credible_set_summary_deduplicated.txt',
    'Deduplicated credible set variants': (
        'credible_set_variants_deduplicated.txt'
    ),
    'Neanderthal-majority signals': 'neanderthal_majority_signals.txt',
    'Kriging diagnostics': 'kriging_flagged_variants.txt',
    'Adaptive introgression overlap': 'adaptive_introgression_overlap.txt',
    'Adaptive introgression variants': (
        'adaptive_introgression_variants.txt'
    ),
    'Reintroduced ancestral overlap': 'reintroduced_ancestral_overlap.txt',
    'Reintroduced ancestral variants': (
        'reintroduced_ancestral_variants.txt'
    ),
}

canonical_inputs = {
    'GWAS metadata': metadata_directory / 'cancer_GWAS_metadata.yaml',
    **{
        name: prefer_current(fine_mapping_directory / file_name)
        for name, file_name in fine_mapping_files.items()
    },
    'Heritability QC': ldsc_directory / 'GWAS_heritability_qc.txt',
    'Heritability enrichments': (
        ldsc_directory / 'introgressed_variants_heritability_summary.txt'
    ),
}


def describe_input(name, path):
    """One inventory row: whether the file exists, when it changed, size."""
    exists = path.exists()
    stat = path.stat() if exists else None
    return {
        'Input': name,
        'File': path.name,
        'Exists': exists,
        'Modified': (
            pd.Timestamp(stat.st_mtime, unit='s').strftime('%Y-%m-%d %H:%M')
            if exists else '—'
        ),
        'Size (MB)': round(stat.st_size / 1e6, 2) if exists else None,
    }


inventory = pd.DataFrame([
    describe_input(name, path) for name, path in canonical_inputs.items()
])
inventory

,Input,File,Exists,Modified,Size (MB)
0,GWAS metadata,cancer_GWAS_metadata.yaml,True,2026-10-31 16:12,0.02
1,Fine-mapping loci,fine_mapping_loci.txt,True,2026-10-25 17:20,0.03
2,Locus fit status,fit_status.txt,True,2026-10-26 04:52,0.01
3,Locus summary,locus_summary.txt,True,2026-10-25 17:54,0.02
4,Credible set variants,credible_set_variants.txt,True,2026-10-25 17:54,1.72
5,Selected credible sets,credible_set_summary_selected.txt,True,2026-10-26 04:52,0.11
6,Deduplicated credible sets,credible_set_summary_deduplicated.txt,True,2026-10-26 04:52,0.11
7,Deduplicated credible set variants,credible_set_variants_deduplicated.txt,True,2026-10-26 04:52,2.18
8,Neanderthal-majority signals,neanderthal_majority_signals.txt,True,2026-10-26 04:52,0.29
9,Kriging diagnostics,kriging_flagged_variants.txt,True,2026-10-25 17:54,0.07


## 2. Cancer GWAS

The GWAS analysed, built from `cancer_GWAS_metadata.yaml` rather than
transcribed, so that the counts reported in Table S2 cannot drift from the
counts the pipeline used.

Not every study enters every analysis. SuSiE needs effect sizes to turn
summary statistics into Z scores, so a study reporting none contributes to the
association-level counts and to the intersection figure, but to no
fine-mapped locus. The `Effect Sizes` column records which.

Studies not deposited in the GWAS Catalog carry placeholder accessions (e.g.
`GCST001`) in the YAML. The code identifies these by requiring six or more
digits after `GCST` and reports them as NA.

`N Effective` is 4 / (1/cases + 1/controls). It is recomputed below and
compared against the stored value.

The output is `gwas_used.txt`, which becomes Table S2 after removing the
effect sizes boolean and manually adding the reported trait and PubMed ID
from the GWAS Catalog.

In [5]:
with open(canonical_inputs['GWAS metadata']) as handle:
    gwas_metadata = yaml.safe_load(handle)

# The metadata stores short cancer keys; the manuscript uses full names.
cancer_names = {
    'BCC': 'Basal Cell Carcinoma',
    'SCC': 'Squamous Cell Carcinoma',
    'bladder': 'Bladder Cancer',
    'breast': 'Breast Cancer',
    'colorectal': 'Colorectal Cancer',
    'endometrial': 'Endometrial Cancer',
    'esophageal': 'Esophageal Cancer',
    'gastric': 'Gastric Cancer',
    'liver': 'Liver Cancer',
    'lung': 'Lung Cancer',
    'melanoma': 'Melanoma',
    'ovarian': 'Ovarian Cancer',
    'pancreatic': 'Pancreatic Cancer',
    'prostate': 'Prostate Cancer',
    'renal': 'Renal Cancer',
    'testicular': 'Testicular Cancer',
    'thyroid': 'Thyroid Cancer',
}

# Publication year by accession; the config key carries only the author.
# Ordered as in Table S2: cancer, then ancestry group.
study_years = {
    'GCST90137411': 2022,  # Basal cell carcinoma, EUR, Seviiri
    'GCST90011817': 2020,  # Bladder, EUR, Rashkin
    'GCST90429846': 2025,  # Breast, EAS, Jia
    'GCST009869': 2019,  # Colorectal, EAS, Lu
    'GCST90255675': 2022,  # Colorectal, EUR, Fernández-Rozadilla
    'GCST90018618': 2021,  # Endometrial, EAS, Sakaue
    'GCST006464': 2018,  # Endometrial, EUR, O'Mara
    'GCST90018621': 2021,  # Esophageal, EAS, Sakaue
    'GCST90011807': 2020,  # Esophageal, EUR, Rashkin
    'GCST90018629': 2021,  # Gastric, EAS, Sakaue
    'GCST90435577': 2018,  # Gastric, EUR, Zhou
    'GCST90018583': 2021,  # Liver, EAS, Sakaue
    'GCST90435580': 2018,  # Liver, EUR, Zhou
    'GCST90018655': 2021,  # Lung, EAS, Sakaue
    'GCST004748': 2017,  # Lung, EUR, McKay
    'GCST90011809': 2020,  # Melanoma, EUR, Rashkin
    'GCST90018668': 2021,  # Ovarian, EAS, Sakaue
    'GCST90011821': 2020,  # Ovarian, EUR, Rashkin
    'GCST90018673': 2021,  # Pancreatic, EAS, Sakaue
    'GCST90011815': 2020,  # Pancreatic, EUR, Rashkin
    'GCST90274716': 2023,  # Prostate, EAS, Wang
    'GCST90274714': 2023,  # Prostate, EUR, Wang
    'GCST90320065': 2024,  # Renal, EAS, Purdue
    'GCST90320057': 2024,  # Renal, EUR, Purdue
    'GCST90137412': 2022,  # Squamous cell carcinoma, EUR, Seviiri
    'GCST90435613': 2018,  # Testicular, EUR, Zhou
    'GCST90018709': 2021,  # Thyroid, EAS, Sakaue
    'GCST90399736': 2022,  # Thyroid, EUR, Zhou
}

gwas_table = pd.DataFrame([
    {
        'Config Key': key,
        'Cancer': cancer_names.get(config.get('cancer'), config.get('cancer')),
        'Ancestry Group': config.get('ancestry_group'),
        'N Cases': config.get('n_cases'),
        'N Controls': config.get('n_controls'),
        'N Effective': config.get('n_effective'),
        'Assembly': config.get('assembly'),
        'Effect Sizes': config.get('effect_sizes'),
    }
    for key, config in gwas_metadata.items()
])

gwas_table['Total N'] = gwas_table['N Cases'] + gwas_table['N Controls']

# Real accessions are GCST plus six or more digits; GCST001 and the like are
# placeholders for studies not deposited in the GWAS Catalog.
gwas_table['GWAS Catalog Accession ID'] = (
    gwas_table['Config Key'].str.extract(r'(GCST\d{6,})$')
)


def format_study(config_key, accession):
    """'Author et al. Year' from the config key and accession."""
    author = config_key.split('_')[-2]
    if accession in study_years:
        return f'{author} et al. {study_years[accession]}'
    return f'{author} et al.'


gwas_table['Study'] = [
    format_study(key, accession)
    for key, accession in zip(
        gwas_table['Config Key'],
        gwas_table['GWAS Catalog Accession ID'].fillna(''),
    )
]

n_eas = (gwas_table['Ancestry Group'] == 'EAS').sum()
n_eur = (gwas_table['Ancestry Group'] == 'EUR').sum()
print(f'{len(gwas_table)} GWAS: {n_eas} EAS, {n_eur} EUR, '
      f'{gwas_table["Cancer"].nunique()} cancers')

with_effects = gwas_table['Effect Sizes'].fillna(False).astype(bool)
print(f'{with_effects.sum()} report effect sizes and entered fine-mapping; '
      f'{(~with_effects).sum()} contribute only to the association-level '
      f'counts and the intersection figure.')

gwas_table.sort_values(['Cancer', 'Ancestry Group'])

29 GWAS: 12 EAS, 17 EUR, 17 cancers
28 report effect sizes and entered fine-mapping; 1 contribute only to the association-level counts and the intersection figure.


,Config Key,Cancer,Ancestry Group,N Cases,N Controls,N Effective,Assembly,Effect Sizes,Total N,GWAS Catalog Accession ID,Study
0,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,Basal Cell Carcinoma,EUR,20791,286893,77544,hg19,True,307684,GCST90137411,Seviiri et al. 2022
1,bladder_cancer_EUR_Rashkin_GCST90011817,Bladder Cancer,EUR,2242,410350,8919,hg19,True,412592,GCST90011817,Rashkin et al. 2020
2,breast_cancer_EAS_Jia_GCST90429846,Breast Cancer,EAS,21319,106116,71010,hg38,True,127435,GCST90429846,Jia et al. 2025
3,breast_cancer_EUR_Zhang_GCST001,Breast Cancer,EUR,133384,113789,245620,hg19,True,247173,NaN,Zhang et al.
4,colorectal_cancer_EAS_Lu_GCST009869,Colorectal Cancer,EAS,23572,48700,63536,hg19,False,72272,GCST009869,Lu et al. 2019
5,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...,Colorectal Cancer,EUR,78473,107143,181188,hg19,True,185616,GCST90255675,Fernandez-Rozadilla et al. 2022
6,endometrial_cancer_EAS_Sakaue_GCST90018618,Endometrial Cancer,EAS,1200,60614,4707,hg19,True,61814,GCST90018618,Sakaue et al. 2021
7,endometrial_cancer_EUR_OMara_GCST006464,Endometrial Cancer,EUR,12906,108979,46158,hg19,True,121885,GCST006464,OMara et al. 2018
8,esophageal_cancer_EAS_Sakaue_GCST90018621,Esophageal Cancer,EAS,1388,159201,5504,hg19,True,160589,GCST90018621,Sakaue et al. 2021
9,esophageal_cancer_EUR_Rashkin_GCST90011807,Esophageal Cancer,EUR,1091,410350,4352,hg19,True,411441,GCST90011807,Rashkin et al. 2020


In [6]:
recomputed_n_effective = (
    4 / (1 / gwas_table['N Cases'] + 1 / gwas_table['N Controls'])
).round()
mismatched = gwas_table.loc[
    (recomputed_n_effective - gwas_table['N Effective']).abs() > 1
]

if len(mismatched):
    display(mismatched.assign(
        **{'N Effective (recomputed)': recomputed_n_effective}
    ))
else:
    print('All n_effective values agree with their case and control counts.')

All n_effective values agree with their case and control counts.


In [7]:
# N SNVs and significant SNVs per GWAS, and the source URL for each summary
# statistics file. Both are keyed on the config key: the URL file's first
# column is the config key plus an `_initial.txt.gz` suffix.
snv_counts = pd.read_csv(
    significant_gwas_variants_directory / 'summaries'
    / 'cancer_GWAS_N_SNVs_and_N_significant_SNVs.txt',
    sep='\t',
).rename(columns={
    'gwas_file': 'Config Key',
    'N_GWAS_SNVs': 'N SNVs',
    'N_significant_GWAS_SNVs': 'N Significant SNVs',
})

urls = pd.read_csv(
    metadata_directory / 'cancer_GWAS_summary_stats_URLs.txt',
    sep='\t',
    header=None,
    names=['File', 'Summary Statistics URL'],
)
urls['Config Key'] = urls['File'].str.replace(
    r'_initial\.txt(\.gz)?$', '', regex=True
)
urls = urls.dropna(subset=['File', 'Summary Statistics URL'])

gwas_summary = (
    gwas_table
    .merge(snv_counts[['Config Key', 'N SNVs', 'N Significant SNVs']],
           on='Config Key', how='left')
    .merge(urls[['Config Key', 'Summary Statistics URL']],
           on='Config Key', how='left')
)

# A GWAS with no SNV count or no URL means one of the three files has fallen
# out of step with the others.
unmatched = gwas_summary[
    gwas_summary[['N SNVs', 'Summary Statistics URL']].isna().any(axis=1)
]
if len(unmatched):
    display(unmatched[['Config Key', 'N SNVs', 'Summary Statistics URL']])
else:
    print('Every GWAS matched an SNV count and a source URL.')

gwas_summary = gwas_summary[[
    'Cancer', 'Ancestry Group', 'GWAS Catalog Accession ID', 'Study',
    'N Cases', 'N Controls', 'Total N', 'N Effective', 'Assembly',
    'N SNVs', 'N Significant SNVs', 'Effect Sizes', 'Summary Statistics URL',
]].sort_values(['Cancer', 'Ancestry Group']).reset_index(drop=True)

gwas_summary_file = supplementary_tables_directory / 'gwas_used.txt'
gwas_summary.to_csv(gwas_summary_file, sep='\t', index=False, na_rep='NA')
print(f'Wrote {len(gwas_summary)} GWAS to {gwas_summary_file}')

gwas_summary

,Config Key,N SNVs,Summary Statistics URL
3,breast_cancer_EUR_Zhang_GCST001,9527352,NaN


Wrote 29 GWAS to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/supplementary_tables/gwas_used.txt


,Cancer,Ancestry Group,GWAS Catalog Accession ID,Study,N Cases,N Controls,Total N,N Effective,Assembly,N SNVs,N Significant SNVs,Effect Sizes,Summary Statistics URL
0,Basal Cell Carcinoma,EUR,GCST90137411,Seviiri et al. 2022,20791,286893,307684,77544,hg19,5301239,8904,True,http://ftp.ebi.ac.uk/pub/databases/gwas/summar...
1,Bladder Cancer,EUR,GCST90011817,Rashkin et al. 2020,2242,410350,412592,8919,hg19,8852977,65,True,http://ftp.ebi.ac.uk/pub/databases/gwas/summar...
2,Breast Cancer,EAS,GCST90429846,Jia et al. 2025,21319,106116,127435,71010,hg38,6559186,1577,True,https://ftp.ebi.ac.uk/pub/databases/gwas/summa...
3,Breast Cancer,EUR,NaN,Zhang et al.,133384,113789,247173,245620,hg19,9527352,18835,True,NaN
4,Colorectal Cancer,EAS,GCST009869,Lu et al. 2019,23572,48700,72272,63536,hg19,8563044,906,False,http://ftp.ebi.ac.uk/pub/databases/gwas/summar...
5,Colorectal Cancer,EUR,GCST90255675,Fernandez-Rozadilla et al. 2022,78473,107143,185616,181188,hg19,11284768,8825,True,https://ftp.ebi.ac.uk/pub/databases/gwas/summa...
6,Endometrial Cancer,EAS,GCST90018618,Sakaue et al. 2021,1200,60614,61814,4707,hg19,12532189,0,True,http://ftp.ebi.ac.uk/pub/databases/gwas/summar...
7,Endometrial Cancer,EUR,GCST006464,OMara et al. 2018,12906,108979,121885,46158,hg19,8514465,410,True,https://ftp.ebi.ac.uk/pub/databases/gwas/summa...
8,Esophageal Cancer,EAS,GCST90018621,Sakaue et al. 2021,1388,159201,160589,5504,hg19,12575417,1042,True,https://ftp.ebi.ac.uk/pub/databases/gwas/summa...
9,Esophageal Cancer,EUR,GCST90011807,Rashkin et al. 2020,1091,410350,411441,4352,hg19,8852133,14,True,http://ftp.ebi.ac.uk/pub/databases/gwas/summar...


## 3. Neanderthal variant maps

Four published maps of Neanderthal-introgressed variants (Browning et al.
2018, Sankararaman et al. 2016, Skov 2024, Vernot et al. 2016) are used
throughout the project. This section counts how many variants each calls,
how they overlap, and how the union partitions across exact combinations —
the basis for the upset plot in Figure S1.

Variants are matched by chromosome and hg19 position (`chr_pos`), as in the
figures notebook. The membership table written here is the upset-plot input.

In [8]:
map_studies = {
    'Browning': 'Browning_et_al_2018',
    'Sankararaman': 'Sankararaman_et_al_2016',
    'Skov': 'Skov_2024',
    'Vernot': 'Vernot_et_al_2016',
}
neanderthal_maps = {
    name: (
        introgressed_variants_directory
        / f'{study}_introgressed_variants_hg19.txt'
    )
    for name, study in map_studies.items()
}


def load_chr_pos(path):
    """Set of 'chr_pos' keys from the first two (headerless) columns."""
    positions = pd.read_csv(
        path, sep='\t', header=None, usecols=[0, 1], names=['chr', 'pos'],
        dtype={'chr': str, 'pos': int},
    )
    return set(positions['chr'] + '_' + positions['pos'].astype(str))


map_calls = {
    name: load_chr_pos(path) for name, path in neanderthal_maps.items()
}

map_summary = (
    pd.DataFrame([
        {'Map': name, 'Variants Called': len(calls)}
        for name, calls in map_calls.items()
    ])
    .sort_values('Variants Called', ascending=False)
    .reset_index(drop=True)
)

union = set().union(*map_calls.values())
print(f'Union of all calls: {len(union):,}')
display(map_summary.style.format({'Variants Called': '{:,}'}))

# One row per chr_pos, one boolean column per map: the R upset plot input.
map_memberships = pd.DataFrame({'chr_pos': sorted(union)})
for name, calls in map_calls.items():
    map_memberships[name] = map_memberships['chr_pos'].isin(calls)

map_memberships_file = (
    introgressed_variants_directory / 'neanderthal_variant_map_memberships.txt'
)
map_memberships.to_csv(map_memberships_file, sep='\t', index=False)
print(f'Wrote {len(map_memberships):,} variants to {map_memberships_file}')

Union of all calls: 1,085,232


,Map,Variants Called
0,Sankararaman,"892,930"
1,Skov,"354,656"
2,Browning,"258,948"
3,Vernot,"232,262"


Wrote 1,085,232 variants to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/introgressed_variants/neanderthal_variant_map_memberships.txt


In [9]:
map_names = list(map_calls)
exact_intersections = []
for size in range(1, len(map_names) + 1):
    for combination in combinations(map_names, size):
        in_all = map_memberships[list(combination)].all(axis=1)
        others = [name for name in map_names if name not in combination]
        in_none_of_others = (
            ~map_memberships[others].any(axis=1) if others else True
        )
        exact_intersections.append({
            'Maps': ', '.join(combination),
            'N Maps': size,
            'Variants': (in_all & in_none_of_others).sum(),
        })

exact_intersections = (
    pd.DataFrame(exact_intersections)
    .sort_values('Variants', ascending=False)
    .reset_index(drop=True)
)
assert exact_intersections['Variants'].sum() == len(union)
exact_intersections

,Maps,N Maps,Variants
0,Sankararaman,1,592508
1,"Browning, Sankararaman, Skov, Vernot",4,95948
2,"Sankararaman, Skov",2,68154
3,Skov,1,66497
4,"Browning, Sankararaman, Skov",3,45987
5,Browning,1,40980
6,"Sankararaman, Vernot",2,31956
7,Vernot,1,27587
8,"Sankararaman, Skov, Vernot",3,25063
9,"Browning, Skov",2,19447


## 4. Neanderthal variant MAFs

Minor allele frequencies from gnomAD v4.1 for the introgressed variant union.
Most introgressed variants are rare in African populations (where
introgression is minimal) but common enough in non-African populations to be
reasonably powered in GWAS. The threshold is MAF > 1% — below it, even the
largest GWAS have little power to detect modest effect sizes.

In [10]:
mafs = pd.read_csv(
    introgressed_variants_directory
    / 'gnomAD_v4.1.0_introgressed_variant_MAFs.txt.gz',
    sep='\t',
    low_memory=False,
    na_values='.',
)
print(f'{len(mafs):,} variants with gnomAD MAF data')

# Labels match the figures notebook.
populations = {
    'MAF_afr': 'African/African American',
    'MAF_eas': 'East Asian',
    'MAF_nfe': 'Non-Finnish European',
    'MAF_sas': 'South Asian',
}
maf_threshold = 0.01

maf_summary = pd.DataFrame([
    {
        'Population': label,
        'Variants': mafs[column].notna().sum(),
        'MAF > 1%': (mafs[column] > maf_threshold).sum(),
        '% MAF > 1%': f'{(mafs[column] > maf_threshold).mean() * 100:.1f}%',
        'Median MAF': f'{mafs[column].median():.4f}',
    }
    for column, label in populations.items()
])

maf_summary

912,978 variants with gnomAD MAF data


,Population,Variants,MAF > 1%,% MAF > 1%,Median MAF
0,African/African American,912978,261580,28.7%,0.0015
1,East Asian,912975,333402,36.5%,0.0014
2,Non-Finnish European,912978,346699,38.0%,0.0016
3,South Asian,912977,438325,48.0%,0.0087


## 5. GWAS loci with Neanderthal variants

For each GWAS, genome-wide significant variants were clumped with PLINK at
several LD thresholds (r² = 0.2, 0.5, 0.8) into independent loci. The same
clumping was repeated using only Neanderthal variants. The ratio — loci
containing at least one Neanderthal variant over total loci — is the basis for
Figures 2B, 2C, S3 and Table S3.

A clump counts as a locus only if it has at least one secondary variant
(PLINK's TOTAL > 0); singleton clumps are excluded because they indicate an
isolated signal rather than a haplotype block where co-occurrence with a
Neanderthal variant is meaningful.

The output is `proportion_loci_with_neanderthal_variants.txt` (Table S3).

In [11]:
all_loci_directory = (
    significant_gwas_variants_directory / 'all_variants' / 'LD_clumped'
)
neanderthal_loci_directory = (
    significant_gwas_variants_directory
    / 'introgressed_variants' / 'LD_clumped'
)

clumped_filename_regex = re.compile(
    r'^(?P<cancer>.*)_(?P<ancestry_group>EUR|EAS)_(?P<author>[A-Za-z0-9\-]+)'
    r'_(?P<gwas_id>GCST[A-Za-z0-9]+)_r2_(?P<r2>[0-9.]+)\.clumped$'
)


def count_loci(path):
    """Clumps with at least one secondary variant.

    PLINK pads .clumped with blank lines, so rows are counted only where CHR
    parses as a number. TOTAL is the count of other variants in the clump, so
    TOTAL == 0 marks a clump resting on its lead variant alone.
    """
    if not path.exists():
        return 0

    clumps = pd.read_csv(path, sep=r'\s+', engine='python')
    if 'CHR' not in clumps.columns or 'TOTAL' not in clumps.columns:
        return 0

    chromosome = pd.to_numeric(clumps['CHR'], errors='coerce')
    total = pd.to_numeric(clumps['TOTAL'], errors='coerce')
    return int((chromosome.notna() & (total > 0)).sum())

In [12]:
rows = []
for all_path in sorted(all_loci_directory.glob('*.clumped')):
    match = clumped_filename_regex.match(all_path.name)
    if match is None:
        print(f'Filename structure did not match: {all_path.name}')
        continue

    fields = match.groupdict()
    n_total = count_loci(all_path)
    n_neanderthal = count_loci(neanderthal_loci_directory / all_path.name)

    rows.append({
        **fields,
        'r2': float(fields['r2']),
        'n_neanderthal': n_neanderthal,
        'n_total': n_total,
        'prop_neanderthal': n_neanderthal / n_total if n_total else 0.0,
    })

proportion_loci_with_neanderthal_variants = pd.DataFrame(rows).sort_values(
    ['ancestry_group', 'cancer', 'r2'], ignore_index=True
)

proportion_loci_file = (
    supplementary_tables_directory
    / 'proportion_loci_with_neanderthal_variants.txt'
)
proportion_loci_with_neanderthal_variants.to_csv(
    proportion_loci_file, sep='\t', index=False
)
print(f'Wrote {len(proportion_loci_with_neanderthal_variants)} rows to '
      f'{proportion_loci_file}')

proportion_loci_with_neanderthal_variants

Wrote 87 rows to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/supplementary_tables/proportion_loci_with_neanderthal_variants.txt


,cancer,ancestry_group,author,gwas_id,r2,n_neanderthal,n_total,prop_neanderthal
0,breast_cancer,EAS,Jia,GCST90429846,0.2,3,41,0.073171
1,breast_cancer,EAS,Jia,GCST90429846,0.5,3,68,0.044118
2,breast_cancer,EAS,Jia,GCST90429846,0.8,5,125,0.040000
3,colorectal_cancer,EAS,Lu,GCST009869,0.2,3,26,0.115385
4,colorectal_cancer,EAS,Lu,GCST009869,0.5,3,34,0.088235
5,colorectal_cancer,EAS,Lu,GCST009869,0.8,3,55,0.054545
6,endometrial_cancer,EAS,Sakaue,GCST90018618,0.2,0,0,0.000000
7,endometrial_cancer,EAS,Sakaue,GCST90018618,0.5,0,0,0.000000
8,endometrial_cancer,EAS,Sakaue,GCST90018618,0.8,0,0,0.000000
9,esophageal_cancer,EAS,Sakaue,GCST90018621,0.2,2,23,0.086957


## 6. Fine-mapping

SuSiE-RSS fine-mapping output (Tables S4, S5 and S13), loaded from the files
written by the Snakemake pipeline. The counts quoted in the Results are
collected here and checked against the manuscript in Section 12.

In [13]:
fine_mapping_loci = pd.read_csv(
    canonical_inputs['Fine-mapping loci'], sep='\t'
)
print(f'{len(fine_mapping_loci)} loci across '
      f'{fine_mapping_loci["cancer"].nunique()} GWAS')
fine_mapping_loci.head()

165 loci across 13 GWAS


,config_key,cancer,ancestry_group,locus_id,lead_rsid,n_samples,chr,pos,effect_allele,other_allele,beta,p_value,neanderthal_allele,introgression_maps,n_maps,window_size,window_start,window_end,n_introgressed_variants
0,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,BCC,EUR,1_150486988,rs72698892,77544,1,150486988,A,G,0.127360,3.640203e-14,NaN,Sankararaman,1,1000,149986988,150986988,68
1,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,BCC,EUR,2_88575445,rs113874908,77544,2,88575445,T,C,-0.150818,5.983079e-17,NaN,Sankararaman,1,1000,88075445,89075445,1
2,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,BCC,EUR,9_16800341,rs2026804,77544,9,16800341,C,A,0.107062,2.885985e-26,C,"Sankararaman, Skov",2,1000,16300341,17300341,48
3,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,BCC,EUR,9_22024351,rs496892,77544,9,22024351,T,C,0.100970,3.657647e-32,NaN,Sankararaman,1,1000,21524351,22524351,2
4,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,BCC,EUR,10_64418739,rs6479830,77544,10,64418739,C,T,0.055391,4.072237e-09,NaN,Sankararaman,1,1000,63918739,64918739,3


In [14]:
fit_status = pd.read_csv(canonical_inputs['Locus fit status'], sep='\t')
n_loci = len(fit_status)
n_converged = fit_status['state'].str.match(
    r'^(converged|rescued)', case=False
).sum()
n_unresolved = n_loci - n_converged

print(f'Loci attempted:       {n_loci}')
print(f'Converged / rescued:  {n_converged}')
print(f'Unresolved:           {n_unresolved}')

Loci attempted:       165
Converged / rescued:  152
Unresolved:           13


In [15]:
selected_credible_sets = pd.read_csv(
    canonical_inputs['Selected credible sets'], sep='\t'
)

n_cs = len(selected_credible_sets)
n_loci_cs = len(
    selected_credible_sets[['cancer', 'ancestry_group', 'locus_id']]
    .drop_duplicates()
)
n_gwas = selected_credible_sets['genome_wide_significant'].eq(True).sum()

print(f'Credible sets (selected):  {n_cs}')
print(f'Unique loci:               {n_loci_cs}')
print(f'Genome-wide significant:   {n_gwas}')

Credible sets (selected):  448
Unique loci:               148
Genome-wide significant:   306


In [16]:
deduplicated_credible_sets = pd.read_csv(
    canonical_inputs['Deduplicated credible sets'], sep='\t'
)

representative = deduplicated_credible_sets[
    deduplicated_credible_sets['is_representative'].eq(True)
]
genome_wide_significant = representative['genome_wide_significant'].eq(True)
has_neanderthal_variant = representative['n_neanderthal'] > 0

n_gwas_dedup = genome_wide_significant.sum()
n_neanderthal_gwas = (genome_wide_significant & has_neanderthal_variant).sum()

print(f'Genome-wide significant:     {n_gwas_dedup}')
print(f'With a Neanderthal variant:  {n_neanderthal_gwas}')

Genome-wide significant:     302
With a Neanderthal variant:  70


In [17]:
kriging = pd.read_csv(canonical_inputs['Kriging diagnostics'], sep='\t')

in_credible_set = kriging['in_credible_set'].eq(True)
is_neanderthal = kriging['is_neanderthal'].eq(True)
severe = kriging['log_lr'] > 10

n_kriging_flagged = len(kriging)
n_kriging_in_cs = in_credible_set.sum()
n_kriging_neanderthal = is_neanderthal.sum()
n_kriging_in_cs_neanderthal = (in_credible_set & is_neanderthal).sum()
n_kriging_severe = severe.sum()

print(f'Kriging-flagged variants:           {n_kriging_flagged}')
print(f'  In a credible set:                {n_kriging_in_cs}')
print(f'  Neanderthal:                      {n_kriging_neanderthal}')
print(f'  Neanderthal and in credible set:  {n_kriging_in_cs_neanderthal}')
print(f'  log(LR) > 10 (severe):            {n_kriging_severe}')

kriging[severe]

Kriging-flagged variants:           520
  In a credible set:                12
  Neanderthal:                      6
  Neanderthal and in credible set:  0
  log(LR) > 10 (severe):            1


,cancer,ancestry_group,locus_id,rsID,CS,pos,z_score,log_lr,is_neanderthal,in_credible_set
12,prostate_cancer_EUR_Wang_GCST90274714,EUR,6_32977535,rs530902542,NaN,32583928,2.314826,10.167697,False,False


## 7. Neanderthal-majority signals

Credible sets in which Neanderthal variants carry more than half of the
posterior mass (Tables S6 and S7).

In [18]:
neanderthal_majority = pd.read_csv(
    canonical_inputs['Neanderthal-majority signals'], sep='\t'
)

n_neanderthal_majority = neanderthal_majority['set_key'].nunique()
print(f'Neanderthal-majority signals (PIP > 50%): {n_neanderthal_majority}')

is_majority = deduplicated_credible_sets['set_key'].isin(
    neanderthal_majority['set_key']
)
majority_summary = deduplicated_credible_sets[is_majority].sort_values(
    'p_neanderthal', ascending=False
)
majority_summary

Neanderthal-majority signals (PIP > 50%): 32


,cancer,ancestry_group,locus_id,lead_rsid,chr,CS,n_variants,n_neanderthal,p_neanderthal,max_pip,max_abs_z,purity,prior_variance,near_window_edge,genome_wide_significant,set_key,signal_id,is_representative
291,prostate_cancer_EUR_Wang_GCST90274714,EUR,2_242254317,rs55659209,2,L4,1,1,1.000000,1.000000,5.533333,1.000000,0.000235,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|2_24...,289,True
279,prostate_cancer_EUR_Wang_GCST90274714,EUR,19_51361757,rs17632542,19,L1,3,3,0.996373,0.458891,27.945946,0.999900,0.002530,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|19_5...,277,True
245,prostate_cancer_EUR_Wang_GCST90274714,EUR,16_57693055,rs16958674,16,L1,2,2,0.989385,0.901123,8.179104,0.976207,0.000119,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|16_5...,243,True
112,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...,EUR,13_73789622,rs61966321,13,L3,10,9,0.962029,0.376259,7.581855,0.920015,0.000237,False,True,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...,111,True
404,renal_cancer_EUR_Purdue_GCST90320057,EUR,7_128580680,rs3807306,7,L1,6,5,0.954026,0.597905,8.244516,0.878959,0.000658,False,True,renal_cancer_EUR_Purdue_GCST90320057|EUR|7_128...,400,True
328,prostate_cancer_EUR_Wang_GCST90274714,EUR,5_52663928,rs55685432,5,L1,25,23,0.936196,0.088431,5.551282,0.704235,0.000057,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|5_52...,326,True
64,breast_cancer_EUR_Zhang_GCST001,EUR,14_68979835,rs11624333,14,L1,8,6,0.906562,0.277961,13.835840,0.884478,0.000735,False,True,breast_cancer_EUR_Zhang_GCST001|EUR|14_6897983...,63,True
297,prostate_cancer_EUR_Wang_GCST90274714,EUR,2_66652885,rs74702681,2,L1,15,12,0.889561,0.367997,7.471910,0.799808,0.000132,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|2_66...,295,True
348,prostate_cancer_EUR_Wang_GCST90274714,EUR,6_28485477,rs114283908,6,L2,49,35,0.842885,0.079744,6.477064,0.930881,0.000100,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|6_28...,346,True
343,prostate_cancer_EUR_Wang_GCST90274714,EUR,6_1682567,rs2814818,6,L1,4,3,0.796869,0.301419,6.524590,0.815415,0.000105,False,True,prostate_cancer_EUR_Wang_GCST90274714|EUR|6_16...,341,True


### Direction of effect

For each Neanderthal-majority association, we orient the GWAS effect size to
the Neanderthal allele. Three of the four introgression maps report a
Neanderthal allele; the Sankararaman CRF map does not. For variants called
only by that map, we take the Neanderthal allele to be the derived allele,
determined from the Ensembl human ancestor sequence (GRCh37, EPO alignment).

Each association is then assigned the direction carrying the greater summed
posterior inclusion probability across its Neanderthal variants. A positive
direction means the Neanderthal haplotype increases cancer risk.

In [19]:
# Fill missing neanderthal_beta for Sankararaman-only variants.
# Those variants have no called Neanderthal allele; the manuscript assumes
# the Neanderthal allele is the derived allele, determined from the Ensembl
# human ancestor sequence (GRCh37, EPO alignment).  Reintroduced ancestral
# alleles are excluded from this assumption.
import pysam

ancestral_fasta = pysam.FastaFile(
    '/wynton/group/capra/data/hg19_ancestral/homo_sapiens_ancestor_GRCh37.fa.gz'
)

neanderthal_vars = neanderthal_majority[
    neanderthal_majority['is_neanderthal'].eq(True)
].copy()

# Identify rows that need filling: neanderthal_beta is NaN
missing = neanderthal_vars['neanderthal_beta'].isna()
print(f'Neanderthal variants missing neanderthal_beta: {missing.sum()} / {len(neanderthal_vars)}')

# Look up the ancestral allele for those positions
for idx in neanderthal_vars.index[missing]:
    chrom = f'chr{neanderthal_vars.at[idx, "chr"]}'
    pos = int(neanderthal_vars.at[idx, 'pos'])
    try:
        ancestral = ancestral_fasta.fetch(chrom, pos - 1, pos).upper()
    except (ValueError, KeyError):
        continue
    if not ancestral or ancestral not in 'ACGT':
        continue

    effect = neanderthal_vars.at[idx, 'effect_allele']
    other = neanderthal_vars.at[idx, 'other_allele']
    beta = neanderthal_vars.at[idx, 'beta']

    # Derived allele = whichever allele is NOT the ancestral allele
    # Neanderthal allele is assumed to be the derived allele
    if ancestral == other:
        # effect allele is derived (= Neanderthal), beta is already correct
        neanderthal_vars.at[idx, 'neanderthal_beta'] = beta
    elif ancestral == effect:
        # other allele is derived (= Neanderthal), flip beta
        neanderthal_vars.at[idx, 'neanderthal_beta'] = -beta
    # else: ancestral doesn't match either allele, leave as NaN

ancestral_fasta.close()

still_missing = neanderthal_vars['neanderthal_beta'].isna().sum()
print(f'Still missing after ancestral lookup: {still_missing}')

# Direction of effect: sum PIP by sign of neanderthal_beta
has_beta = neanderthal_vars['neanderthal_beta'].notna()
directed = neanderthal_vars[has_beta].copy()
directed['direction'] = np.where(
    directed['neanderthal_beta'] > 0, 'risk', 'protective'
)

pip_by_direction = (
    directed
    .groupby(['set_key', 'direction'])['PIP']
    .sum()
    .unstack(fill_value=0)
)

signal_direction = pd.Series(
    np.where(
        pip_by_direction.get('risk', 0) >= pip_by_direction.get('protective', 0),
        'risk-increasing',
        'risk-decreasing',
    ),
    index=pip_by_direction.index,
    name='direction',
)

n_risk = (signal_direction == 'risk-increasing').sum()
n_protective = (signal_direction == 'risk-decreasing').sum()

print(f'\nRisk-increasing:  {n_risk}')
print(f'Risk-decreasing:  {n_protective}')
print(f'Total:            {n_risk + n_protective}')

Neanderthal variants missing neanderthal_beta: 307 / 579
Still missing after ancestral lookup: 0

Risk-increasing:  14
Risk-decreasing:  18
Total:            32


In [20]:
direction_summary = (
    majority_summary
    .assign(
        direction=lambda df: df['set_key'].map(signal_direction),
        cancer=lambda df: clean_cancer_names(
            df['cancer'], df['ancestry_group']
        ),
    )
    .sort_values(['direction', 'cancer', 'p_neanderthal'], ascending=[True, True, False])
)

direction_summary[[
    'cancer', 'ancestry_group', 'locus_id', 'CS',
    'p_neanderthal', 'max_abs_z', 'direction',
]]

,cancer,ancestry_group,locus_id,CS,p_neanderthal,max_abs_z,direction
17,basal_cell_carcinoma,EUR,12_52908254,L2,0.770039,10.519119,risk-decreasing
64,breast_cancer,EUR,14_68979835,L1,0.906562,13.835840,risk-decreasing
59,breast_cancer,EUR,12_28153018,L3,0.789165,15.519034,risk-decreasing
41,breast_cancer,EAS,12_28143787,L1,0.632709,7.911243,risk-decreasing
57,breast_cancer,EUR,12_28153018,L2,0.541407,8.853790,risk-decreasing
131,colorectal_cancer,EUR,19_33519927,L1,0.672020,11.130595,risk-decreasing
291,prostate_cancer,EUR,2_242254317,L4,1.000000,5.533333,risk-decreasing
279,prostate_cancer,EUR,19_51361757,L1,0.996373,27.945946,risk-decreasing
245,prostate_cancer,EUR,16_57693055,L1,0.989385,8.179104,risk-decreasing
328,prostate_cancer,EUR,5_52663928,L1,0.936196,5.551282,risk-decreasing


## 8. Adaptive introgression

Distance from each Neanderthal-majority credible set to the nearest
high-frequency ancient-and-modern introgressed region from Iasi et al. 2025.
The output is `adaptive_introgression_distances.txt` (Table S8).

In [21]:
neanderthal_majority_cs_coords = (
    neanderthal_majority
    .assign(chr_bed=lambda df: 'chr' + df['chr'].astype(str))
    .groupby('set_key')
    .agg(chr=('chr_bed', 'first'), start=('pos', 'min'), end=('pos', 'max'))
    .reset_index()
)

neanderthal_majority_cs_bed = pbt.BedTool.from_dataframe(
    neanderthal_majority_cs_coords[['chr', 'start', 'end', 'set_key']]
).sort()

ai_regions_file = (
    adaptive_introgression_directory
    / 'Iasi_et_al_2025_high_frequency_ancient_modern_regions.bed'
)
ai_regions_bed = pbt.BedTool(str(ai_regions_file)).sort()
print(f'Iasi et al. AI regions: {ai_regions_bed.count()}')

Iasi et al. AI regions: 177


In [22]:
closest_ai_regions = neanderthal_majority_cs_bed.closest(
    ai_regions_bed, d=True
)

ai_distances = pd.DataFrame([
    {
        'set_key': hit[3],
        'chr': hit[0],
        'start': int(hit[1]),
        'end': int(hit[2]),
        'ai_chr': hit[4],
        'ai_start': int(hit[5]),
        'ai_end': int(hit[6]),
        'distance_bp': int(hit[-1]),
    }
    for hit in closest_ai_regions
])

ai_distances[['cancer', 'ancestry_group', 'locus_id', 'CS']] = (
    ai_distances['set_key'].str.split('|', expand=True)
)
ai_distances['cancer'] = clean_cancer_names(
    ai_distances['cancer'], ai_distances['ancestry_group']
)
ai_distances['distance_kb'] = ai_distances['distance_bp'] / 1000

ai_distances = ai_distances[[
    'cancer', 'ancestry_group', 'locus_id', 'CS', 'chr', 'start', 'end',
    'ai_chr', 'ai_start', 'ai_end', 'distance_bp', 'distance_kb',
]].sort_values('distance_kb')

n_in_ai = (ai_distances['distance_bp'] == 0).sum()
print(f'In an AI region:     {n_in_ai} / {len(ai_distances)}')
print(f'Not in an AI region: {len(ai_distances) - n_in_ai}')

ai_distances_file = (
    supplementary_tables_directory / 'adaptive_introgression_distances.txt'
)
ai_distances.to_csv(ai_distances_file, sep='\t', index=False)
print(f'Wrote {len(ai_distances)} credible sets to {ai_distances_file}')

ai_distances

In an AI region:     5 / 32
Not in an AI region: 27
Wrote 32 credible sets to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/supplementary_tables/adaptive_introgression_distances.txt


,cancer,ancestry_group,locus_id,CS,chr,start,end,ai_chr,ai_start,ai_end,distance_bp,distance_kb
0,squamous_cell_carcinoma,EUR,1_40035928,L1,chr1,39564930,40088043,chr1,39390316,40075608,0,0.000
8,basal_cell_carcinoma,EUR,12_52908254,L2,chr12,52895301,52913517,chr12,52691034,53054592,0,0.000
29,basal_cell_carcinoma,EUR,9_16800341,L2,chr9,16795286,16800789,chr9,16711940,16804382,0,0.000
31,colorectal_cancer,EUR,9_113661528,L1,chr9,113652080,113677755,chr9,113336588,113916949,0,0.000
30,squamous_cell_carcinoma,EUR,9_16800341,L2,chr9,16795286,16800789,chr9,16711940,16804382,0,0.000
16,colorectal_cancer,EUR,19_33519927,L1,chr19,33517152,33520961,chr19,33531058,33707363,10098,10.098
5,breast_cancer,EUR,12_28153018,L3,chr12,28139733,28182252,chr12,28193086,28200255,10835,10.835
6,breast_cancer,EAS,12_28143787,L1,chr12,28140253,28170174,chr12,28193086,28200255,22913,22.913
10,breast_cancer,EUR,14_68979835,L1,chr14,68974410,69029203,chr14,69058317,69067171,29115,29.115
7,breast_cancer,EUR,12_28153018,L2,chr12,28318354,28446889,chr12,28193086,28200255,118100,118.100


## 9. Locus examples

Deep dives into five Neanderthal-majority credible sets: *BNC2*,
*KRT5*/*KRT14*, *PABPC4*, *SLC24A4* and *PTHLH*. Open Targets associations
for *BNC2* and *SLC24A4* are in Tables S9 and S10.

In [23]:
dedup_variants = pd.read_csv(
    canonical_inputs['Deduplicated credible set variants'], sep='\t'
)
dedup_variants['cancer'] = clean_cancer_names(
    dedup_variants['cancer'], dedup_variants['ancestry_group']
)
dedup_variants = dedup_variants.drop(columns=[
    'set_key',
    'genome_wide_significant.x',
    'genome_wide_significant.y',
    'signal_id',
    'is_representative',
])

dedup_variants.head()

,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
0,basal_cell_carcinoma,EUR,10_64418739,L1,rs6479830,10,0.009856,rs10995255,64402433,A,G,0.112212,0.110105,0.050706,5.719953,0.964871,False,NaN,NaN,NaN,NaN
1,basal_cell_carcinoma,EUR,10_64418739,L1,rs6479830,10,0.009856,rs10995256,64402434,T,G,0.099000,0.096869,0.050531,5.698870,0.964871,False,NaN,NaN,NaN,NaN
2,basal_cell_carcinoma,EUR,10_64418739,L1,rs6479830,10,0.009856,rs10995248,64396042,T,C,0.089851,0.087706,0.050217,5.696428,0.964871,False,NaN,NaN,NaN,NaN
3,basal_cell_carcinoma,EUR,10_64418739,L1,rs6479830,10,0.009856,rs10995246,64391846,A,C,0.087840,0.085672,0.049961,5.667234,0.964871,False,NaN,NaN,NaN,NaN
4,basal_cell_carcinoma,EUR,10_64418739,L1,rs6479830,10,0.009856,rs10995249,64396916,T,C,0.087427,0.085278,0.050179,5.691784,0.964871,False,NaN,NaN,NaN,NaN


### 9.1 *BNC2*

In [24]:
bnc2_locus = '9_16800341'
bnc2_variant = 'rs2026805'

bnc2_scc_cs = get_credible_set(
    dedup_variants, 'squamous_cell_carcinoma', bnc2_locus
)
bnc2_bcc_cs = get_credible_set(
    dedup_variants, 'basal_cell_carcinoma', bnc2_locus
)

for label, credible_set in [('SCC', bnc2_scc_cs), ('BCC', bnc2_bcc_cs)]:
    beta = credible_set.loc[credible_set['rsID'] == bnc2_variant, 'beta']
    print(f'{label}: {len(credible_set)} variants; '
          f'{bnc2_variant} beta = {beta.item():.3f}')

bnc2_scc_cs

SCC: 10 variants; rs2026805 beta = 0.063
BCC: 66 variants; rs2026805 beta = 0.104


,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
6958,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs10810650,16873551,T,C,0.373111,0.372133,0.074975,13.597565,0.973749,False,NaN,NaN,NaN,NaN
6959,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs10962650,16874612,T,C,0.230132,0.228936,0.074812,13.563092,0.973749,False,NaN,NaN,NaN,NaN
6960,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs10810652,16876283,A,G,0.141710,0.140376,0.074577,13.501740,0.973749,False,NaN,NaN,NaN,NaN
6961,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs2153271,16864521,T,C,0.128462,0.127091,0.075000,13.623987,0.973749,False,NaN,NaN,NaN,NaN
6962,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs62541919,16863364,C,T,0.080580,0.079138,0.074713,13.548105,0.973749,False,NaN,NaN,NaN,NaN
6963,squamous_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,2.422433e-19,rs1416742,16856883,A,G,0.053809,0.052321,0.074609,13.558876,0.973749,False,NaN,NaN,NaN,NaN
6964,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,7.772941e-01,rs2026805,16800789,A,G,0.621068,0.620464,0.062559,9.969386,0.975591,True,"Sankararaman, Skov",A,0.062559,NaN
6965,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,7.772941e-01,rs2026804,16800341,C,A,0.154352,0.153029,0.063762,10.087366,0.975591,True,"Sankararaman, Skov",C,0.063762,NaN
6966,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,7.772941e-01,rs10962599,16795286,T,C,0.148419,0.147072,0.064489,10.207265,0.975591,False,NaN,NaN,NaN,NaN
6967,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,7.772941e-01,rs10962605,16799623,G,A,0.044645,0.043165,0.063620,10.068134,0.975591,False,NaN,NaN,NaN,NaN


In [25]:
bnc2_bcc_cs

,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
291,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs2153271,16864521,T,C,0.282353,0.282167,0.123255,14.010347,0.973749,False,NaN,NaN,NaN,NaN
292,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs10810650,16873551,T,C,0.255915,0.255722,0.121331,13.761549,0.973749,False,NaN,NaN,NaN,NaN
293,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs62541919,16863364,C,T,0.207532,0.207327,0.123312,13.984045,0.973749,False,NaN,NaN,NaN,NaN
294,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs10810652,16876283,A,G,0.123058,0.122831,0.121159,13.708985,0.973749,False,NaN,NaN,NaN,NaN
295,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs10962650,16874612,T,C,0.076314,0.076077,0.120721,13.673651,0.973749,False,NaN,NaN,NaN,NaN
296,basal_cell_carcinoma,EUR,9_16800341,L1,rs2026804,9,7.363789e-22,rs1416742,16856883,A,G,0.056102,0.055859,0.122496,13.911009,0.973749,False,NaN,NaN,NaN,NaN
297,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs2026805,16800789,A,G,0.402597,0.402431,0.104086,10.383325,0.975591,True,"Sankararaman, Skov",A,0.104086,NaN
298,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs2026804,16800341,C,A,0.214911,0.214700,0.107062,10.602991,0.975591,True,"Sankararaman, Skov",C,0.107062,NaN
299,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs10962605,16799623,G,A,0.197631,0.197415,0.107633,10.661664,0.975591,False,NaN,NaN,NaN,NaN
300,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs10962599,16795286,T,C,0.105204,0.104965,0.107347,10.634210,0.975591,False,NaN,NaN,NaN,NaN


In [26]:
chr9_ccres_file = annotations_directory / 'GRCh37-cCREs_chr9.bed'

bnc2_scc_cres = intersect_with_cres(bnc2_scc_cs, chr9_ccres_file)
bnc2_scc_cres

,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes,cre_chr,cre_start,cre_end,dhs_accession,ccre_accession,ccre_class
0,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,0.777294,rs2026805,16800789,A,G,0.621068,0.620464,0.062559,9.969386,0.975591,True,"Sankararaman, Skov",A,0.062559,NaN,chr9,16800623,16800841,EH38D4175877,EH38E3876183,dELS
1,squamous_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,0.777294,rs10962599,16795286,T,C,0.148419,0.147072,0.064489,10.207265,0.975591,False,NaN,NaN,NaN,NaN,chr9,16795266,16795616,EH38D4175874,EH38E2684646,dELS


In [27]:
bnc2_bcc_cres = intersect_with_cres(bnc2_bcc_cs, chr9_ccres_file)
bnc2_bcc_cres

,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes,cre_chr,cre_start,cre_end,dhs_accession,ccre_accession,ccre_class
0,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs2026805,16800789,A,G,0.402597,0.402431,0.104086,10.383325,0.975591,True,"Sankararaman, Skov",A,0.104086,NaN,chr9,16800623,16800841,EH38D4175877,EH38E3876183,dELS
1,basal_cell_carcinoma,EUR,9_16800341,L2,rs2026804,9,6.331737e-01,rs10962599,16795286,T,C,0.105204,0.104965,0.107347,10.634210,0.975591,False,NaN,NaN,NaN,NaN,chr9,16795266,16795616,EH38D4175874,EH38E2684646,dELS
2,basal_cell_carcinoma,EUR,9_16800341,L3,rs2026804,9,4.576845e-10,rs7856477,16602654,A,G,0.020707,0.020406,-0.097052,-5.380667,0.757733,False,NaN,NaN,NaN,NaN,chr9,16602601,16602822,EH38D4175682,EH38E2684494,dELS
3,basal_cell_carcinoma,EUR,9_16800341,L3,rs2026804,9,4.576845e-10,rs1927617,16572642,T,C,0.004584,0.004296,-0.089959,-5.093947,0.757733,False,NaN,NaN,NaN,NaN,chr9,16572382,16572721,EH38D6015046,EH38E3876087,dELS
4,basal_cell_carcinoma,EUR,9_16800341,L7,rs2026804,9,1.785843e-03,rs10962459,16501503,G,A,0.033618,0.033367,-0.052706,-5.078407,0.632950,False,NaN,NaN,NaN,NaN,chr9,16501218,16501566,EH38D6014995,EH38E3876047,dELS
5,basal_cell_carcinoma,EUR,9_16800341,L7,rs2026804,9,1.785843e-03,rs7041549,16501402,G,T,0.025504,0.025253,-0.052387,-5.062359,0.632950,False,NaN,NaN,NaN,NaN,chr9,16501218,16501566,EH38D6014995,EH38E3876047,dELS
6,basal_cell_carcinoma,EUR,9_16800341,L7,rs2026804,9,1.785843e-03,rs17811225,16526653,T,C,0.003204,0.002904,-0.036844,-3.000451,0.632950,False,NaN,NaN,NaN,NaN,chr9,16526407,16526755,EH38D4175601,EH38E2684437,dELS


Open Targets associations (Table S9). All lead variants on the Neanderthal
haplotype (in our credible sets or in high LD with rs2026805, r² ≥ 0.88) have
their alt allele on that haplotype, so betas read directly as Neanderthal
effects: lighter skin, more blonde and less dark brown/black hair, and higher
risk of SCC, actinic keratosis, melanoma and other skin cancers. The glaucoma
(rs6475067) and hypertension (rs1330298) leads are in weak LD (r² = 0.15 and
0.31) and likely reflect independent signals, so they are excluded.

In [28]:
bnc2_open_targets = pd.read_csv(
    open_targets_directory / 'open_targets_bnc2_cs.txt', sep='\t'
)
summarize_phenotypes(bnc2_open_targets)

,Mapped Trait,Reported Trait,n_studies,beta,min_p,direction
10,skin pigmentation,Skin colour (UKB data field 1717),2,-0.041291,4.751308e-111,decrease
13,strand of hair color,"Hair colour (natural, before greying): Blonde ...",1,0.016776,3.732200e-33,increase
1,actinic keratosis,Actinic keratosis (PheCode 702.1),1,0.021254,2.901000e-32,increase
6,skin cancer,Skin cancer,2,0.019564,3.438000e-30,increase
9,skin neoplasm,Other non-epithelial cancer of skin (PheCode 1...,1,0.016215,1.340000e-28,increase
7,skin disorder,Degenerative skin conditions and other dermato...,1,0.013276,3.602000e-19,increase
2,cutaneous melanoma,"Melanomas of skin, dx or hx (PheCode 172.1)",1,0.012615,4.296000e-18,increase
3,hair color,Hair colour natural before greying - Dark brow...,3,-0.012308,5.957611e-18,decrease
14,strand of hair color,"Hair colour (natural, before greying): Dark br...",1,-0.011170,1.773715e-16,decrease
12,strand of hair color,"Hair colour (natural, before greying): Black (...",1,-0.010007,4.435426e-13,decrease


### 9.2 *KRT* cluster

In [29]:
krt_bcc_cs = get_credible_set(
    dedup_variants, 'basal_cell_carcinoma', '12_52908254'
)
beta = krt_bcc_cs.loc[krt_bcc_cs['rsID'] == 'rs687751', 'beta'].item()
print(f'rs687751 beta = {beta:.3f}')
krt_bcc_cs

rs687751 beta = -0.100


,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
58,basal_cell_carcinoma,EUR,12_52908254,L1,rs639790,12,5.369453e-14,rs11170164,52913668,T,C,0.696403,0.695855,0.175083,11.091065,0.939379,False,NaN,NaN,NaN,NaN
59,basal_cell_carcinoma,EUR,12_52908254,L1,rs639790,12,5.369453e-14,rs11611584,52906035,A,C,0.302755,0.301457,0.163551,10.987428,0.939379,False,NaN,NaN,NaN,NaN
60,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs639790,52908254,T,C,0.219742,0.218343,-0.100491,-10.519119,0.970037,True,Sankararaman,NaN,NaN,NaN
61,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs687751,52907803,G,A,0.185040,0.183582,-0.100204,-10.499980,0.970037,True,Sankararaman,NaN,NaN,NaN
62,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs1798659,52896368,G,T,0.070939,0.069297,-0.099428,-10.424369,0.970037,False,NaN,NaN,NaN,NaN
63,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs641556,52896742,C,T,0.070425,0.068782,-0.099432,-10.423535,0.970037,True,Sankararaman,NaN,NaN,NaN
64,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs627835,52897503,G,A,0.068773,0.067128,-0.099396,-10.420813,0.970037,True,Sankararaman,NaN,NaN,NaN
65,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs627881,52897470,C,T,0.068772,0.067126,-0.099394,-10.420810,0.970037,True,Sankararaman,NaN,NaN,NaN
66,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs830381,52899240,T,C,0.063948,0.062295,-0.099301,-10.412453,0.970037,True,Sankararaman,NaN,NaN,NaN
67,basal_cell_carcinoma,EUR,12_52908254,L2,rs639790,12,7.700394e-01,rs11170152,52898835,C,T,0.054091,0.052420,-0.099260,-10.409031,0.970037,False,NaN,NaN,NaN,NaN


### 9.3 *PABPC4*

In [30]:
pabpc4_scc_cs = get_credible_set(
    dedup_variants, 'squamous_cell_carcinoma', '1_40035928'
)
beta = pabpc4_scc_cs.loc[pabpc4_scc_cs['rsID'] == 'rs3768321', 'beta'].item()
print(f'rs3768321 beta = {beta:.3f}')
pabpc4_scc_cs

rs3768321 beta = 0.041


,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
6873,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs3768321,40035928,T,G,0.334227,0.330791,0.040851,6.064575,0.579216,True,"Browning, Sankararaman",T,0.040851,NaN
6874,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs72663520,40003410,T,C,0.049818,0.045542,0.035928,5.642184,0.579216,False,NaN,NaN,NaN,NaN
6875,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs17513135,40035686,T,C,0.047258,0.042934,0.035789,5.618869,0.579216,True,"Browning, Sankararaman",T,0.035789,NaN
6876,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs4660293,40028180,G,A,0.043565,0.039276,0.035437,5.623903,0.579216,True,"Browning, Sankararaman",G,0.035437,NaN
6877,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs755249,39995074,T,C,0.037677,0.033376,0.035405,5.593714,0.579216,True,Browning,T,0.035405,NaN
6878,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs58302493,40033905,A,G,0.036630,0.031603,0.041672,5.666048,0.579216,True,Sankararaman,NaN,NaN,NaN
6879,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs61779359,40010235,C,T,0.032020,0.027802,0.035296,5.548309,0.579216,True,Browning,C,0.035296,NaN
6880,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs61781363,40012433,T,C,0.031691,0.027473,0.035268,5.546012,0.579216,False,NaN,NaN,NaN,NaN
6881,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs61781364,40019591,G,A,0.030789,0.026578,0.035201,5.537011,0.579216,False,NaN,NaN,NaN,NaN
6882,squamous_cell_carcinoma,EUR,1_40035928,L1,rs3768321,1,0.732129,rs61781362,40012230,T,C,0.030724,0.026509,0.035239,5.539116,0.579216,False,NaN,NaN,NaN,NaN


### 9.4 *SLC24A4*

In [31]:
slc24a4_scc_cs = get_credible_set(
    dedup_variants, 'squamous_cell_carcinoma', '14_92773295'
)
beta = slc24a4_scc_cs.loc[
    slc24a4_scc_cs['rsID'] == 'rs79342051', 'beta'
].item()
print(f'rs79342051 beta = {beta:.3f}')
slc24a4_scc_cs

rs79342051 beta = -0.056


,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes
6578,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs1957283,92767680,G,A,0.129264,0.127713,0.057027,6.200551,0.800665,False,NaN,NaN,NaN,NaN
6579,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs12435365,92806211,G,A,0.060039,0.057722,-0.057275,-5.555283,0.800665,True,Sankararaman,NaN,NaN,NaN
6580,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs79342051,92767097,T,G,0.057647,0.056083,-0.055781,-5.951850,0.800665,True,"Browning, Sankararaman",T,-0.055781,NaN
6581,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs8021744,92773295,C,T,0.055197,0.053636,0.056216,5.990607,0.800665,True,Sankararaman,NaN,NaN,NaN
6582,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs10135799,92789548,G,T,0.051287,0.049673,0.056045,6.010943,0.800665,False,NaN,NaN,NaN,NaN
6583,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs746589,92775389,A,G,0.050141,0.048576,0.056103,5.975677,0.800665,True,Sankararaman,NaN,NaN,NaN
6584,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs76369916,92767473,T,C,0.048822,0.047251,-0.055537,-5.926038,0.800665,True,"Browning, Sankararaman",T,-0.055537,NaN
6585,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs76021903,92787216,G,A,0.043247,0.041677,-0.056126,-5.985600,0.800665,True,Sankararaman,NaN,NaN,NaN
6586,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs79291676,92774503,T,C,0.042025,0.040455,-0.055713,-5.948027,0.800665,True,Browning,T,-0.055713,NaN
6587,squamous_cell_carcinoma,EUR,14_92773295,L1,rs8021744,14,7.942290e-01,rs12436973,92773250,G,A,0.040418,0.038847,-0.055653,-5.941882,0.800665,True,Browning,G,-0.055653,NaN


Open Targets associations (Table S10). Leads are on the non-Neanderthal
haplotype, so **flip the signs**. The Neanderthal haplotype is associated with
higher IGF-1, lower smoking initiation and lower SCC risk (consistent with the
negative Neanderthal allele betas in Table S7). The height hit is a separate
signal 1.8 Mb away and is excluded.

In [32]:
slc24a4_open_targets = pd.read_csv(
    open_targets_directory / 'open_targets_slc24a4_cs.txt', sep='\t'
)
summarize_phenotypes(slc24a4_open_targets)

,Mapped Trait,Reported Trait,n_studies,beta,min_p,direction
3,IGF-1 measurement,Insulin-like growth factor 1 levels,1,0.010418,1.760000e-10,increase
1,IGF-1 measurement,IGF-1 levels,2,-0.009580,2.564000e-10,decrease
0,IGF-1 measurement,IGF 1 (UKB data field 30770),2,-0.073066,2.875000e-10,decrease
6,squamous cell carcinoma,Squamous cell carcinoma (MTAG),1,0.057027,5.627000e-10,increase
2,IGF-1 measurement,IGF-1 levels (UKB data field 30770),1,-0.022241,5.991000e-10,decrease
5,smoking initiation,Smoking initiation,1,0.007570,1.000000e-09,increase
4,body height,"Height, inverse-rank normalized",1,0.007763,2.010000e-08,increase


### 9.5 *PTHLH*

Breast cancer credible sets in the East Asian and European GWAS. The second
European locus at this region (`12_28658479`) contains the same signals from
an overlapping fine-mapping window, so only `12_28153018` is used.

In [33]:
pthlh_loci = {'EAS': '12_28143787', 'EUR': '12_28153018'}

pthlh_credible_sets = pd.concat([
    get_credible_set(dedup_variants, 'breast_cancer', locus_id)
    for locus_id in pthlh_loci.values()
])

shared_variant = 'rs788464'  # in the EAS credible set and European CS3
pthlh_credible_sets.loc[
    pthlh_credible_sets['rsID'] == shared_variant,
    ['ancestry_group', 'CS', 'beta'],
]

,ancestry_group,CS,beta
572,EAS,L1,-0.127900
641,EUR,L3,-0.149599


In [34]:
chr12_ccres_file = annotations_directory / 'GRCh37-cCREs_chr12.bed'

pthlh_cres = intersect_with_cres(pthlh_credible_sets, chr12_ccres_file)
pthlh_cres

,cancer,ancestry_group,locus_id,CS,lead_rsid,chr,p_neanderthal,rsID,pos,effect_allele,other_allele,PIP,alpha,beta,Z,purity,is_neanderthal,introgression_maps,neanderthal_allele,neanderthal_beta,notes,cre_chr,cre_start,cre_end,dhs_accession,ccre_accession,ccre_class
0,breast_cancer,EAS,12_28143787,L1,rs1267205,12,6.327087e-01,rs805513,28148641,G,A,0.021739,0.017782,0.128300,7.682635,0.958229,True,Browning,A,-0.128300,NaN,chr12,28148377,28148680,EH38D4680443,EH38E3009073,dELS
1,breast_cancer,EUR,12_28153018,L1,rs788464,12,1.763489e-24,rs7297051,28174817,T,C,0.983181,0.983053,-0.123410,-17.339109,1.000000,False,NaN,NaN,NaN,NaN,chr12,28174675,28175025,EH38D4680457,EH38E3009083,dELS
2,breast_cancer,EUR,12_28153018,L2,rs788464,12,5.414073e-01,rs11612143,28355627,C,T,0.142939,0.136360,-0.057424,-8.851428,0.958093,False,NaN,NaN,NaN,NaN,chr12,28355337,28355674,EH38D4680547,EH38E3009147,"dELS,CTCF-bound"
3,breast_cancer,EUR,12_28153018,L2,rs788464,12,5.414073e-01,rs17801436,28318354,C,T,0.075949,0.068894,-0.056759,-8.726319,0.958093,True,Browning,C,-0.056759,NaN,chr12,28318293,28318541,EH38D4680525,EH38E3009135,pELS
4,breast_cancer,EUR,12_28153018,L3,rs788464,12,7.891651e-01,rs16932559,28181233,C,T,0.105607,0.097445,-0.146944,-15.388845,0.754906,False,NaN,NaN,NaN,NaN,chr12,28180990,28181338,EH38D4680465,EH38E3009089,dELS
5,breast_cancer,EUR,12_28153018,L3,rs788464,12,7.891651e-01,rs805513,28148641,G,A,0.052708,0.045233,0.148375,15.363595,0.754906,True,Browning,A,-0.148375,NaN,chr12,28148377,28148680,EH38D4680443,EH38E3009073,dELS
6,breast_cancer,EUR,12_28153018,L3,rs788464,12,7.891651e-01,rs11049293,28179701,C,T,0.015039,0.007446,-0.148708,-15.482897,0.754906,False,NaN,NaN,NaN,NaN,chr12,28179664,28180012,EH38D4680462,EH38E3009087,dELS
7,breast_cancer,EUR,12_28153018,L3,rs788464,12,7.891651e-01,rs12230582,28179075,T,C,0.014167,0.006633,-0.149049,-15.347320,0.754906,True,"Browning, Sankararaman",T,-0.149049,NaN,chr12,28178810,28179146,EH38D2542775,EH38E1601544,dELS


#### Credible set summary

One row per credible set, written to `pthlh_credible_set_summary.txt` for the
table panel in the *PTHLH* figure. The lead variant is the one with the
highest PIP; the Neanderthal beta is the largest-magnitude Neanderthal allele
effect in the set.

In [35]:
def summarize_credible_set(credible_set):
    """Lead variant, size and Neanderthal content of one credible set."""
    lead = credible_set.loc[credible_set['PIP'].idxmax()]
    neanderthal = credible_set[credible_set['is_neanderthal'].eq(True)]
    neanderthal_betas = neanderthal['neanderthal_beta'].dropna()
    return pd.Series({
        'lead_variant': lead['rsID'],
        'position_mb': lead['pos'] / 1e6,
        'n_variants': len(credible_set),
        'n_neanderthal': len(neanderthal),
        'p_neanderthal': credible_set['p_neanderthal'].iloc[0],
        'neanderthal_beta': (
            neanderthal_betas.loc[neanderthal_betas.abs().idxmax()]
            if len(neanderthal_betas) else np.nan
        ),
        'max_abs_z': credible_set['Z'].abs().max(),
    })


pthlh_summary = (
    pthlh_credible_sets
    .groupby(['ancestry_group', 'locus_id', 'CS'])[[
        'rsID', 'pos', 'PIP', 'is_neanderthal', 'neanderthal_beta',
        'p_neanderthal', 'Z',
    ]]
    .apply(summarize_credible_set)
    .reset_index()
    .sort_values(['ancestry_group', 'CS'])
)

pthlh_summary_file = (
    supplementary_tables_directory / 'pthlh_credible_set_summary.txt'
)
pthlh_summary.to_csv(pthlh_summary_file, sep='\t', index=False)
print(f'Wrote {len(pthlh_summary)} credible sets to {pthlh_summary_file}')

pthlh_summary

Wrote 4 credible sets to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/supplementary_tables/pthlh_credible_set_summary.txt


,ancestry_group,locus_id,CS,lead_variant,position_mb,n_variants,n_neanderthal,p_neanderthal,neanderthal_beta,max_abs_z
0,EAS,12_28143787,L1,rs1267211,28.140829,34,27,6.327087e-01,-0.133600,7.911243
1,EUR,12_28153018,L1,rs7297051,28.174817,1,0,1.763489e-24,NaN,17.339109
2,EUR,12_28153018,L2,rs10843134,28.358643,9,5,5.414073e-01,-0.057372,8.853790
3,EUR,12_28153018,L3,rs788464,28.153018,28,22,7.891651e-01,-0.150150,15.519034


#### Conditional analysis

We condition the European and East Asian breast cancer windows at *PTHLH* on
the strongest European association, rs7297051, using
$z_{cond} = (z - r z_{lead}) / \sqrt{1 - r^2}$, to test whether the nearby
credible sets are independent signals or one signal spread by LD.

In [36]:
conditional_directory = fine_mapping_directory / 'conditional'
conditioning_variant = 'rs7297051'


def read_conditional(ancestry):
    """The conditioned window for one ancestry group."""
    pattern = (
        f'breast_cancer_{ancestry}_*_conditioned_on_{conditioning_variant}.txt'
    )
    paths = sorted(conditional_directory.glob(pattern))
    assert len(paths) == 1, f'Expected one {ancestry} file, found {paths}'
    return pd.read_csv(paths[0], sep='\t')


conditional = pd.concat(
    {ancestry: read_conditional(ancestry) for ancestry in pthlh_loci},
    names=['ancestry_group'],
).reset_index(level='ancestry_group').reset_index(drop=True)

conditional.groupby('ancestry_group').size()

ancestry_group
EAS    1935
EUR     343
dtype: int64

Check the conditional Z-scores against the formula.

In [37]:
def recompute_conditional_z(window, scale=1.0):
    """Conditional Z on the conditioning variant; `scale` rescales the Zs."""
    z = window['Z'] * scale
    z_lead = z[window['rsID'] == conditioning_variant].item()
    return (z - window['r'] * z_lead) / np.sqrt(1 - window['r'] ** 2)


conditional['z_recomputed'] = conditional.groupby(
    'ancestry_group', group_keys=False
)[['rsID', 'Z', 'r']].apply(recompute_conditional_z)

not_conditioning_variant = conditional['rsID'] != conditioning_variant
max_difference = (
    conditional.loc[not_conditioning_variant, 'z_conditional']
    - conditional.loc[not_conditioning_variant, 'z_recomputed']
).abs().max()
print(f'Largest difference from the formula: {max_difference:.2e}')

Largest difference from the formula: 1.49e-13


Conditional signal in each credible set. Genome-wide significance is
$P = 5 \times 10^{-8}$, two-sided. The attenuation ratio is the conditional
$|Z|$ divided by the marginal $|Z|$ for the credible set's lead variant
(highest PIP, as in the summary table above).

In [38]:
z_threshold = NormalDist().inv_cdf(1 - 5e-8 / 2)
print(f'Genome-wide |Z| threshold: {z_threshold:.2f}')

pthlh_conditional = conditional.merge(
    pthlh_credible_sets[['ancestry_group', 'locus_id', 'CS', 'rsID', 'PIP']],
    on=['ancestry_group', 'rsID'],
)

conditional_summary = (
    pthlh_conditional
    .sort_values('PIP', ascending=False)
    .groupby(['ancestry_group', 'locus_id', 'CS'])
    .agg(
        n_variants=('rsID', 'size'),
        lead_variant=('rsID', 'first'),
        marginal_z=('Z', 'first'),
        conditional_z=('z_conditional', 'first'),
        min_abs_conditional_z=('z_conditional', lambda z: z.abs().min()),
        max_abs_conditional_z=('z_conditional', lambda z: z.abs().max()),
        median_abs_r=('r', lambda r: r.abs().median()),
    )
)
conditional_summary['attenuation_ratio'] = (
    conditional_summary['conditional_z'].abs()
    / conditional_summary['marginal_z'].abs()
)
conditional_summary['genome_wide_significant'] = (
    conditional_summary['max_abs_conditional_z'] >= z_threshold
)
conditional_summary.round(2)

Genome-wide |Z| threshold: 5.45


n_variants lead_variant  marginal_z  \
ancestry_group locus_id    CS                                        
EAS            12_28143787 L1          34    rs1267211        7.91   
EUR            12_28153018 L1           1    rs7297051      -17.34   
                           L2           9   rs10843134       -8.83   
                           L3          28     rs788464      -15.52   

                               conditional_z  min_abs_conditional_z  \
ancestry_group locus_id    CS                                         
EAS            12_28143787 L1           3.48                   2.73   
EUR            12_28153018 L1            NaN                    NaN   
                           L2          -8.49                   8.17   
                           L3          -6.60                   5.62   

                               max_abs_conditional_z  median_abs_r  \
ancestry_group locus_id    CS                                        
EAS            12_28143787 L1                   3.48          0.86   
EUR            12_28153018 L1                    NaN          1.00   
                           L2                   8.51          0.02   
                           L3                   6.60          0.59   

                               attenuation_ratio  genome_wide_significant  
ancestry_group locus_id    CS                                              
EAS            12_28143787 L1               0.44                    False  
EUR            12_28153018 L1                NaN                    False  
                           L2               0.96                     True  
                           L3               0.43                     True

Rescaling the East Asian statistics to European power. At a variant shared by
both credible sets (rs788464), $Z$ per unit effect size measures each GWAS's
power. We rescale the East Asian Zs by the European/East Asian ratio and
recompute the conditional statistic with $r$ held fixed. This assumes equal
effect sizes across ancestries and is a power illustration, not a formal
test.

In [39]:
shared = conditional.query('rsID == @shared_variant').set_index(
    'ancestry_group'
)
z_per_beta = shared['Z'] / shared['beta']
power_ratio = z_per_beta['EUR'] / z_per_beta['EAS']
print(f'Power ratio at {shared_variant}: {power_ratio:.2f}')

eas_window = conditional.query("ancestry_group == 'EAS'").copy()
eas_window['z_conditional_rescaled'] = recompute_conditional_z(
    eas_window, power_ratio
)
eas_rescaled = eas_window.merge(
    pthlh_credible_sets[['ancestry_group', 'rsID']],
    on=['ancestry_group', 'rsID'],
)
rescaled_max = eas_rescaled['z_conditional_rescaled'].abs().max()
print(f'Max rescaled conditional |Z| in the EAS credible set: '
      f'{rescaled_max:.2f} (threshold {z_threshold:.2f})')

Power ratio at rs788464: 1.74
Max rescaled conditional |Z| in the EAS credible set: 6.06 (threshold 5.45)


#### Reintroduced ancestral alleles

In [40]:
ancestral_variants = pd.read_csv(
    canonical_inputs['Reintroduced ancestral variants'], sep='\t'
)
ancestral_variants['cancer'] = clean_cancer_names(
    ancestral_variants['cancer'], ancestral_variants['ancestry_group']
)

pthlh_ancestral = ancestral_variants[
    (ancestral_variants['cancer'] == 'breast_cancer')
    & ancestral_variants['locus_id'].isin(pthlh_loci.values())
]

pthlh_raa_summary = (
    pthlh_ancestral
    .groupby(['ancestry_group', 'locus_id', 'CS'])
    .agg(
        cs_size=('rsID', 'size'),
        n_neanderthal=('is_neanderthal', 'sum'),
        n_reintroduced_ancestral=('is_reintroduced_ancestral', 'sum'),
    )
    .reset_index()
)

pthlh_raa_summary

,ancestry_group,locus_id,CS,cs_size,n_neanderthal,n_reintroduced_ancestral
0,EAS,12_28143787,L1,34,27,13
1,EUR,12_28153018,L1,1,0,0
2,EUR,12_28153018,L2,9,5,0
3,EUR,12_28153018,L3,28,22,7


In [41]:
is_raa = pthlh_ancestral['is_reintroduced_ancestral'].eq(True)
pthlh_ancestral.loc[
    is_raa, ['ancestry_group', 'CS', 'rsID', 'chr', 'pos', 'is_neanderthal']
]

,ancestry_group,CS,rsID,chr,pos,is_neanderthal
549,EAS,L1,rs1267211,12,28140829,False
551,EAS,L1,rs1267210,12,28140918,True
554,EAS,L1,rs1267213,12,28140675,False
556,EAS,L1,rs251955,12,28143462,True
557,EAS,L1,rs1267206,12,28143712,True
559,EAS,L1,rs788458,12,28149568,True
561,EAS,L1,rs788461,12,28150488,False
564,EAS,L1,rs251952,12,28145950,True
567,EAS,L1,rs805513,12,28148641,True
569,EAS,L1,rs2737448,12,28153260,True


## 10. Ancient DNA

Allele frequency of rs2026804 through time from ancient DNA genotype calls
(AADR), binned by sample date. The output, `rs2026804_binned_DAFs.txt`, is
plotted in the figures notebook.

- Bin widths grow with age because sample density falls off: at this locus,
  after deduplication, there are 821 specimens in 0–2 ka, 539 in 2–5, 267 in
  5–10, 19 in 10–20 and 11 in 20–50.
- Present-day samples are held out as their own category rather than folded
  into the youngest bin. They outnumber the entire ancient series two to one,
  and they are a worldwide anthropological panel rather than a sample of any
  one population, so their regional frequencies are not comparable to the
  ancient bins or to gnomAD.
- One genotype is kept per specimen. Shotgun.diploid is preferred because it
  is the only source here that can call a heterozygote; the remaining sources
  are pseudo-haploid, so a true heterozygote is reported as one homozygote or
  the other. Among those the ordering is arbitrary and only breaks ties where
  both libraries carry a call.
- Allele counts are ploidy-aware. A pseudo-haploid specimen contributes one
  allele, not two: counting it as two leaves the frequency unchanged but
  understates the binomial variance by a factor of √2, which matters for the
  oldest bins where the whole claim rests on a handful of specimens.
- Intervals are Wilson score intervals. The Wald interval collapses to zero
  width when no derived alleles are observed, which is precisely the oldest
  bins.

In [42]:
DATE_BIN_BOUNDS = [0, 2000, 5000, 10000, 20000, 50000]
SOURCE_PRIORITY = ['Shotgun.diploid', 'Shotgun', '3.7M', '1240K']
GENOTYPE_DOSAGES = {'0/0': 0, '0/1': 1, '1/0': 1, '1/1': 2}


def bin_ancient_genotypes(path, conf_level=0.95):
    """Derived allele frequency and Wilson interval by date bin and region."""
    z = NormalDist().inv_cdf(1 - (1 - conf_level) / 2)

    # Labels run from the older to the younger bound, e.g. '[5-2)'. Bins include
    # their older bound, so a specimen dated exactly 2 ka is in [2-0).
    age_labels = [
        f'[{older / 1000:g}-{younger / 1000:g})'
        for younger, older in zip(DATE_BIN_BOUNDS[:-1], DATE_BIN_BOUNDS[1:])
    ]
    bin_levels = age_labels[::-1] + ['Present-day']

    genotypes = pd.read_csv(path, na_values=['', 'NA', 'NaN', './.'])
    genotypes['GT'] = genotypes['Genotype'].map(GENOTYPE_DOSAGES)
    # Shotgun.diploid is the only diploid source: every heterozygous call in
    # the file comes from it.
    genotypes['ploidy'] = np.where(
        genotypes['Data_source'] == 'Shotgun.diploid', 2, 1
    )
    genotypes['source_rank'] = (
        genotypes['Data_source']
        .map({source: rank for rank, source in enumerate(SOURCE_PRIORITY)})
        .fillna(len(SOURCE_PRIORITY))
    )
    # Right-closed bins exclude Date_mean == 0, which is present-day.
    age_bin = pd.cut(
        genotypes['Date_mean'], bins=DATE_BIN_BOUNDS, labels=age_labels,
        right=True,
    ).astype(object)
    genotypes['date_bin'] = age_bin.where(
        genotypes['Date_mean'] != 0, 'Present-day'
    )
    genotypes = genotypes.dropna(
        subset=['GT', 'date_bin', 'Region', 'Specimen_ID']
    )

    deduplicated = (
        genotypes
        .sort_values(['Specimen_ID', 'source_rank'], kind='mergesort')
        .drop_duplicates('Specimen_ID')
        .copy()
    )
    deduplicated['observed_alleles'] = deduplicated['ploidy']
    deduplicated['derived_observed'] = np.where(
        deduplicated['ploidy'] == 2, deduplicated['GT'], deduplicated['GT'] / 2
    )

    # A pseudo-haploid call is 0 or 2, never 1. If this fires, the ploidy
    # assignment is wrong and the allele counts cannot be trusted.
    heterozygous_haploid = (
        (deduplicated['ploidy'] == 1) & (deduplicated['GT'] == 1)
    ).sum()
    if heterozygous_haploid:
        raise ValueError(
            f'{heterozygous_haploid} pseudo-haploid call(s) are heterozygous; '
            'check the ploidy assignment.'
        )

    present_day = deduplicated['date_bin'] == 'Present-day'
    print(f'{len(genotypes)} calls from {genotypes["Specimen_ID"].nunique()} '
          f'specimens; {len(genotypes) - len(deduplicated)} duplicate '
          f'libraries dropped. {present_day.sum()} specimens are present-day '
          f'and {(~present_day).sum()} ancient; '
          f'{(deduplicated["ploidy"] == 2).sum()} of {len(deduplicated)} are '
          f'diploid.')

    binned = (
        deduplicated
        .groupby(['date_bin', 'Region'])
        .agg(
            sample_size=('Specimen_ID', 'size'),
            total_alleles=('observed_alleles', 'sum'),
            derived_alleles=('derived_observed', 'sum'),
        )
        .reset_index()
    )
    n = binned['total_alleles']
    frequency = binned['derived_alleles'] / n
    centre = (frequency + z**2 / (2 * n)) / (1 + z**2 / n)
    half_width = (
        z * np.sqrt(frequency * (1 - frequency) / n + z**2 / (4 * n**2))
        / (1 + z**2 / n)
    )
    binned['allele_frequency'] = frequency
    binned['ci_low'] = (centre - half_width).clip(lower=0)
    binned['ci_high'] = (centre + half_width).clip(upper=1)

    # bin_order lets the figure restore the display order, oldest first.
    binned['bin_order'] = binned['date_bin'].map(
        {label: order for order, label in enumerate(bin_levels)}
    )
    return binned.sort_values(['bin_order', 'Region'], ignore_index=True)

In [43]:
rs2026804_binned_dafs = bin_ancient_genotypes(
    ancient_genomes_directory / 'rs2026804_AADR_genotypes.csv'
)

binned_dafs_file = ancient_genomes_directory / 'rs2026804_binned_DAFs.txt'
rs2026804_binned_dafs.to_csv(binned_dafs_file, sep='\t', index=False)
print(f'Wrote {len(rs2026804_binned_dafs)} bins to {binned_dafs_file}')

rs2026804_binned_dafs

8045 calls from 5364 specimens; 2681 duplicate libraries dropped. 3713 specimens are present-day and 1651 ancient; 3580 of 5364 are diploid.
Wrote 31 bins to /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/ancient_genomes/rs2026804_binned_DAFs.txt


,date_bin,Region,sample_size,total_alleles,derived_alleles,allele_frequency,ci_low,ci_high,bin_order
0,[50-20),East Asia,3,4,0.0,0.000000,0.000000e+00,0.489891,0
1,[50-20),Europe,8,8,0.0,0.000000,0.000000e+00,0.324408,0
2,[20-10),Africa,2,2,0.0,0.000000,0.000000e+00,0.657620,1
3,[20-10),Americas,6,6,0.0,0.000000,0.000000e+00,0.390334,1
4,[20-10),Europe,9,9,6.0,0.666667,3.542021e-01,0.879416,1
5,[20-10),West Asia,2,2,2.0,1.000000,3.423802e-01,1.000000,1
6,[10-5),Africa,3,4,0.0,0.000000,0.000000e+00,0.489891,2
7,[10-5),Americas,8,8,0.0,0.000000,0.000000e+00,0.324408,2
8,[10-5),East Asia,36,36,1.0,0.027778,4.920407e-03,0.141697,2
9,[10-5),Europe,185,187,129.0,0.689840,6.202807e-01,0.751756,2


## 11. Heritability

LD score regression heritability and partitioned heritability enrichment in
Neanderthal variants (Tables S11 and S12). A GWAS enters the enrichment
analysis only if its mean χ² is at least 1.02 and its observed-scale h² is at
least 0.05; enrichments estimated on less signal are not interpretable. The
QC table, with the reason for each exclusion, is written to
`heritability_qc.txt`, which the figures notebook reads to filter the
enrichment panels.

In [44]:
min_mean_chi2 = 1.02
min_h2 = 0.05

heritability_qc = pd.read_csv(canonical_inputs['Heritability QC'], sep='\t')
heritability_qc['reason_not_included'] = np.select(
    [
        heritability_qc['mean_chi2'] < min_mean_chi2,
        heritability_qc['h2_observed'] < min_h2,
    ],
    [
        f'Mean chi-square below {min_mean_chi2}',
        f'Heritability below {min_h2}',
    ],
    default='',
)
heritability_qc['included'] = heritability_qc['reason_not_included'] == ''

print(f'{heritability_qc["included"].sum()} of {len(heritability_qc)} GWAS '
      f'pass QC for the enrichment analysis')

heritability_qc_file = supplementary_tables_directory / 'heritability_qc.txt'
heritability_qc.to_csv(heritability_qc_file, sep='\t', index=False)
print(f'Wrote {heritability_qc_file}')

heritability_qc

8 of 28 GWAS pass QC for the enrichment analysis
Wrote /wynton/group/capra/projects/neanderthal_ancestry_and_cancer_risk/data/supplementary_tables/heritability_qc.txt


,cancer,ancestry_group,study,n_cases,n_controls,n_effective,n_snps,lambda_GC,mean_chi2,intercept,intercept_SE,h2_observed,h2_observed_SE,sample_prevalence,population_prevalence,h2_liability,h2_liability_SE,reason_not_included,included
0,BCC,EUR,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,20791,286893,77544,1022321,1.1428,1.4089,0.9302,0.0204,0.0795,0.0094,NaN,NaN,NaN,NaN,,True
1,SCC,EUR,squamous_cell_carcinoma_EUR_Seviiri_GCST90137412,7402,286892,28863,1022321,1.0895,1.3609,0.7568,0.0159,0.1048,0.0086,NaN,NaN,NaN,NaN,,True
2,bladder,EUR,bladder_cancer_EUR_Rashkin_GCST90011817,2242,410350,8919,1163050,1.0165,1.0282,1.0030,0.0080,0.0035,0.0019,NaN,NaN,NaN,NaN,Heritability below 0.05,False
3,breast,EAS,breast_cancer_EAS_Jia_GCST90429846,21319,106116,71010,1026699,1.0895,1.1262,0.9897,0.0104,0.0634,0.0077,NaN,NaN,NaN,NaN,,True
4,breast,EUR,breast_cancer_EUR_Zhang_GCST001,133384,113789,245620,1128010,1.3650,1.6746,1.0772,0.0165,0.1381,0.0092,NaN,NaN,NaN,NaN,,True
5,colorectal,EUR,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...,78473,107143,181188,1170427,1.2966,1.4700,1.0855,0.0150,0.1218,0.0098,NaN,NaN,NaN,NaN,,True
6,endometrial,EAS,endometrial_cancer_EAS_Sakaue_GCST90018618,1200,60614,4707,1043632,1.0135,1.0102,0.9860,0.0083,0.0129,0.0122,NaN,NaN,NaN,NaN,Mean chi-square below 1.02,False
7,endometrial,EUR,endometrial_cancer_EUR_OMara_GCST006464,12906,108979,46158,1169686,1.1082,1.1255,1.0455,0.0095,0.0371,0.0075,NaN,NaN,NaN,NaN,Heritability below 0.05,False
8,esophageal,EAS,esophageal_cancer_EAS_Sakaue_GCST90018621,1388,159201,5504,1043800,1.0255,1.0338,0.9772,0.0158,0.0199,0.0123,NaN,NaN,NaN,NaN,Heritability below 0.05,False
9,esophageal,EUR,esophageal_cancer_EUR_Rashkin_GCST90011807,1091,410350,4352,1163014,1.0165,1.0107,1.0016,0.0076,0.0009,0.0017,NaN,NaN,NaN,NaN,Mean chi-square below 1.02,False


In [45]:
heritability_enrichment = pd.read_csv(
    canonical_inputs['Heritability enrichments'], sep='\t'
)

# The QC is keyed on the GWAS config key (`study`). Older summaries append
# the Neanderthal map to the key in `gwas`, so that suffix is stripped.
map_suffix = '_(' + '|'.join(map_studies.values()) + ')$'
heritability_enrichment['study'] = heritability_enrichment['gwas'].str.replace(
    map_suffix, '', regex=True
)

unmatched_gwas = (
    set(heritability_enrichment['study']) - set(heritability_qc['study'])
)
if unmatched_gwas:
    print(f'No QC row for: {", ".join(sorted(unmatched_gwas))}')

included_studies = heritability_qc.loc[heritability_qc['included'], 'study']
heritability_enrichment[
    heritability_enrichment['study'].isin(included_studies)
]

,gwas,variant_set,Prop._SNPs,Prop._h2,Prop._h2_std_error,Enrichment,Enrichment_std_error,Enrichment_p,n_snps,h2,h2_se,lambda_gc,mean_chi2,intercept,intercept_se,study
0,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,Browning_et_al_2018,0.008710,0.003691,0.005800,0.423773,0.665901,0.379798,1022315,0.0799,0.0095,1.1428,1.4089,0.9287,0.0212,basal_cell_carcinoma_EUR_Seviiri_GCST90137411
2,breast_cancer_EAS_Jia_GCST90429846,Browning_et_al_2018,0.011183,0.026052,0.013161,2.329659,1.176961,0.247956,1026667,0.0622,0.0077,1.0895,1.1262,0.9914,0.0104,breast_cancer_EAS_Jia_GCST90429846
3,breast_cancer_EUR_Zhang_GCST001,Browning_et_al_2018,0.008710,0.014931,0.008674,1.714376,0.995883,0.475686,1127915,0.1369,0.0088,1.3650,1.6746,1.0811,0.0174,breast_cancer_EUR_Zhang_GCST001
4,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...,Browning_et_al_2018,0.008710,0.007871,0.003301,0.903734,0.378984,0.800076,1170394,0.1221,0.0100,1.2966,1.4700,1.0849,0.0152,colorectal_cancer_EUR_Fernandez-Rozadilla_GCST...
14,lung_cancer_EUR_McKay_GCST004748,Browning_et_al_2018,0.008710,0.006675,0.007498,0.766409,0.860900,0.787757,1106060,0.0785,0.0113,1.1523,1.1695,1.0520,0.0090,lung_cancer_EUR_McKay_GCST004748
20,prostate_cancer_EAS_Wang_GCST90274716,Browning_et_al_2018,0.011183,0.011267,0.008581,1.007567,0.767383,0.992094,1068281,0.0845,0.0101,1.0710,1.1448,0.9928,0.0125,prostate_cancer_EAS_Wang_GCST90274716
21,prostate_cancer_EUR_Wang_GCST90274714,Browning_et_al_2018,0.008710,0.006192,0.004011,0.710961,0.460506,0.533389,1186611,0.0756,0.0072,1.4069,1.9992,1.0971,0.0447,prostate_cancer_EUR_Wang_GCST90274714
24,squamous_cell_carcinoma_EUR_Seviiri_GCST90137412,Browning_et_al_2018,0.008710,0.009996,0.004826,1.147713,0.554068,0.784593,1022315,0.1043,0.0087,1.0895,1.3609,0.7586,0.0165,squamous_cell_carcinoma_EUR_Seviiri_GCST90137412
28,basal_cell_carcinoma_EUR_Seviiri_GCST90137411,Sankararaman_et_al_2016,0.031445,0.016737,0.012214,0.532261,0.388433,0.229337,1022315,0.0800,0.0096,1.1428,1.4089,0.9280,0.0215,basal_cell_carcinoma_EUR_Seviiri_GCST90137411
30,breast_cancer_EAS_Jia_GCST90429846,Sankararaman_et_al_2016,0.036160,0.047459,0.016863,1.312455,0.466343,0.497807,1026667,0.0627,0.0077,1.0895,1.1262,0.9907,0.0103,breast_cancer_EAS_Jia_GCST90429846


## 12. Manuscript values

Every number quoted in the manuscript, beside the value the pipeline outputs
give now. Each entry is (manuscript value, tolerance); approximate values
("≈") get a looser tolerance. Update the manuscript or this list whenever a
✗ appears.

In [46]:
manuscript_values = {
    # Section 6. Fine-mapping
    'n_loci': (165, 0),
    'n_converged': (152, 0),
    'n_loci_cs': (148, 0),
    'n_cs': (448, 0),
    'n_gwas': (306, 0),
    'n_gwas_dedup': (302, 0),
    'n_neanderthal_gwas': (70, 0),
    'n_kriging_flagged': (520, 0),
    'n_kriging_in_cs': (12, 0),
    'n_kriging_neanderthal': (6, 0),
    'n_kriging_in_cs_neanderthal': (0, 0),
    'n_kriging_severe': (1, 0),
    # Section 7. Neanderthal-majority signals
    'n_neanderthal_majority': (32, 0),
    # Section 7. Direction of effect
    'n_risk_increasing': (14, 0),
    'n_risk_decreasing': (18, 0),
    # Section 9.5. PTHLH conditional analysis
    'pthlh_z_threshold': (5.45, 0.005),
    'pthlh_eur_cs2_conditional_z': (8.51, 0.005),
    'pthlh_eur_cs3_conditional_z': (6.60, 0.005),
    'pthlh_eas_min_conditional_z': (2.7, 0.05),
    'pthlh_eas_max_conditional_z': (3.5, 0.05),
    'pthlh_eas_median_abs_r': (0.85, 0.02),
    'pthlh_eur_cs3_median_abs_r': (0.60, 0.02),
    'pthlh_eas_attenuation_ratio': (0.44, 0.005),
    'pthlh_eur_cs3_attenuation_ratio': (0.43, 0.005),
    'pthlh_power_ratio': (1.74, 0.005),
    'pthlh_eas_rescaled_conditional_z': (6.1, 0.05),
}

eas_cs = conditional_summary.loc[('EAS', pthlh_loci['EAS'], 'L1')]
eur_cs2 = conditional_summary.loc[('EUR', pthlh_loci['EUR'], 'L2')]
eur_cs3 = conditional_summary.loc[('EUR', pthlh_loci['EUR'], 'L3')]

pipeline_values = {
    'n_loci': n_loci,
    'n_converged': n_converged,
    'n_loci_cs': n_loci_cs,
    'n_cs': n_cs,
    'n_gwas': n_gwas,
    'n_gwas_dedup': n_gwas_dedup,
    'n_neanderthal_gwas': n_neanderthal_gwas,
    'n_kriging_flagged': n_kriging_flagged,
    'n_kriging_in_cs': n_kriging_in_cs,
    'n_kriging_neanderthal': n_kriging_neanderthal,
    'n_kriging_in_cs_neanderthal': n_kriging_in_cs_neanderthal,
    'n_kriging_severe': n_kriging_severe,
    'n_neanderthal_majority': n_neanderthal_majority,
    'n_risk_increasing': n_risk,
    'n_risk_decreasing': n_protective,
    'pthlh_z_threshold': z_threshold,
    'pthlh_eur_cs2_conditional_z': eur_cs2['max_abs_conditional_z'],
    'pthlh_eur_cs3_conditional_z': eur_cs3['max_abs_conditional_z'],
    'pthlh_eas_min_conditional_z': eas_cs['min_abs_conditional_z'],
    'pthlh_eas_max_conditional_z': eas_cs['max_abs_conditional_z'],
    'pthlh_eas_median_abs_r': eas_cs['median_abs_r'],
    'pthlh_eur_cs3_median_abs_r': eur_cs3['median_abs_r'],
    'pthlh_eas_attenuation_ratio': eas_cs['attenuation_ratio'],
    'pthlh_eur_cs3_attenuation_ratio': eur_cs3['attenuation_ratio'],
    'pthlh_power_ratio': power_ratio,
    'pthlh_eas_rescaled_conditional_z': rescaled_max,
}

for name, (manuscript_value, tolerance) in manuscript_values.items():
    pipeline_value = pipeline_values[name]
    matches = abs(pipeline_value - manuscript_value) <= tolerance
    print(f'{"✓" if matches else "✗"}  {name}: '
          f'manuscript={manuscript_value}, pipeline={pipeline_value:.4g}')

✓  n_loci: manuscript=165, pipeline=165
✓  n_converged: manuscript=152, pipeline=152
✓  n_loci_cs: manuscript=148, pipeline=148
✓  n_cs: manuscript=448, pipeline=448
✓  n_gwas: manuscript=306, pipeline=306
✓  n_gwas_dedup: manuscript=302, pipeline=302
✓  n_neanderthal_gwas: manuscript=70, pipeline=70
✓  n_kriging_flagged: manuscript=520, pipeline=520
✓  n_kriging_in_cs: manuscript=12, pipeline=12
✓  n_kriging_neanderthal: manuscript=6, pipeline=6
✓  n_kriging_in_cs_neanderthal: manuscript=0, pipeline=0
✓  n_kriging_severe: manuscript=1, pipeline=1
✓  n_neanderthal_majority: manuscript=32, pipeline=32
✓  n_risk_increasing: manuscript=14, pipeline=14
✓  n_risk_decreasing: manuscript=18, pipeline=18
✓  pthlh_z_threshold: manuscript=5.45, pipeline=5.451
✓  pthlh_eur_cs2_conditional_z: manuscript=8.51, pipeline=8.505
✓  pthlh_eur_cs3_conditional_z: manuscript=6.6, pipeline=6.604
✓  pthlh_eas_min_conditional_z: manuscript=2.7, pipeline=2.73
✓  pthlh_eas_max_conditional_z: manuscript=3.5, pip

## 13. Session information

In [47]:
print(f'Python {platform.python_version()}')
for module in [np, pd, pbt, yaml]:
    print(f'{module.__name__} {module.__version__}')

Python 3.14.7
numpy 2.5.2
pandas 3.0.5
pybedtools 0.12.1
yaml 6.0.3
